# YOLOv8s Helmet Detection Training
**Dataset:** Kaggle hard-hat-detection + Roboflow PPE (~6176 images)  
**Model:** YOLOv8s pretrained on ImageNet  
**Result:** mAP50 = 94.9%

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import subprocess, os, shutil, random, json
import xml.etree.ElementTree as ET
from pathlib import Path

subprocess.run(["pip", "install", "ultralytics", "kaggle", "-q"])
os.makedirs("/root/.kaggle", exist_ok=True)

# ⚠️ Kaggle API bilgilerinizi buraya girin
# kaggle.com/settings > API > Create New Token ile indirin
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump({"username": "YOUR_KAGGLE_USERNAME", "key": "YOUR_KAGGLE_API_KEY"}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
# Kaggle hard-hat-detection dataset'ini indir
os.makedirs("/content/kaggle_raw", exist_ok=True)
subprocess.run(["kaggle", "datasets", "download", "-d", "andrewmvd/hard-hat-detection",
                "--unzip", "-p", "/content/kaggle_raw", "-q"])

for split in ["train", "valid", "test"]:
    os.makedirs("/content/merged/" + split + "/images", exist_ok=True)
    os.makedirs("/content/merged/" + split + "/labels", exist_ok=True)

CLASS_MAP = {"helmet": 0, "head": 1}

def get_yolo_lines(xml_path, W, H):
    lines = []
    for obj in ET.parse(xml_path).getroot().findall("object"):
        name = obj.find("name").text.lower().strip()
        if name not in CLASS_MAP:
            continue
        bb = obj.find("bndbox")
        xmin = float(bb.find("xmin").text)
        ymin = float(bb.find("ymin").text)
        xmax = float(bb.find("xmax").text)
        ymax = float(bb.find("ymax").text)
        cx = round((xmin + xmax) / 2 / W, 6)
        cy = round((ymin + ymax) / 2 / H, 6)
        bw = round((xmax - xmin) / W, 6)
        bh = round((ymax - ymin) / H, 6)
        lines.append(str(CLASS_MAP[name]) + " " + str(cx) + " " + str(cy) + " " + str(bw) + " " + str(bh))
    return lines

all_items = []

for xml_path in Path("/content/kaggle_raw/annotations").glob("*.xml"):
    img_path = "/content/kaggle_raw/images/" + xml_path.stem + ".png"
    if not os.path.exists(img_path):
        continue
    root = ET.parse(str(xml_path)).getroot()
    size = root.find("size")
    if size is None:
        continue
    W = int(size.find("width").text)
    H = int(size.find("height").text)
    if W == 0 or H == 0:
        continue
    lines = get_yolo_lines(str(xml_path), W, H)
    if lines:
        all_items.append(("kgl", img_path, lines))

# ⚠️ Drive'daki Roboflow PPE dataset yolunu kendinize göre güncelleyin
OLD = "/content/drive/Othercomputers/MacBook Pro'm/Desktop/ppe detection.yolov8/train"

for img in Path(OLD + "/images").glob("*.*"):
    if img.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue
    lbl = Path(OLD + "/labels") / (img.stem + ".txt")
    if not lbl.exists():
        continue
    with open(str(lbl)) as f:
        content = f.read().strip()
    if content:
        all_items.append(("old", str(img), content.split("\n")))

print("Toplam: " + str(len(all_items)))

random.seed(42)
random.shuffle(all_items)
n = len(all_items)
boundaries = [0, int(n * 0.7), int(n * 0.9), n]
split_names = ["train", "valid", "test"]

for i, split in enumerate(split_names):
    chunk = all_items[boundaries[i]:boundaries[i+1]]
    for item in chunk:
        src_type, img_path, lines = item
        fname = Path(img_path).stem
        ext = Path(img_path).suffix
        shutil.copy(img_path, "/content/merged/" + split + "/images/" + fname + ext)
        lbl_path = "/content/merged/" + split + "/labels/" + fname + ".txt"
        with open(lbl_path, "w") as f:
            f.write("\n".join(lines))
    print(split + ": " + str(len(chunk)))

shutil.rmtree("/content/kaggle_raw")
print("Veri hazir!")

In [ ]:
import shutil
from ultralytics import YOLO

with open("/content/data.yaml", "w") as f:
    f.write("train: /content/merged/train/images\n")
    f.write("val: /content/merged/valid/images\n")
    f.write("test: /content/merged/test/images\n")
    f.write("nc: 2\n")
    f.write("names: ['Helmet', 'No_Helmet']\n")

model = YOLO("yolov8s.pt")
model.train(data="/content/data.yaml", epochs=50, imgsz=640, batch=16,
            patience=10, cls=2.0, name="helmet_v3")

# ⚠️ Kaydedilecek Drive yolunu kendinize göre güncelleyin
shutil.copy("/content/runs/detect/helmet_v3/weights/best.pt",
            "/content/drive/MyDrive/helmet_v3_best.pt")
print("Kaydedildi!")